In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
import torch.nn as nn
from pyro.infer import Predictive

###PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

###Forward and Backward Selection
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
import statsmodels.api as sm

# HMC
from pyro.infer import MCMC, NUTS

# variational inference
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from tqdm.auto import trange
from tqdm.notebook import trange

import matplotlib as mpl
import os
import sys
import math

C:\Users\thumo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
### Setting Direction for Optbnn file
sys.path.append(
    r"C:\Users\thumo\OneDrive - Georgia Institute of Technology\Georgia Tech\Semesters\Spring 2025\CSE 8803 IUQ\Project\2-BNN_trained_prior\you-need-a-good-prior"
)

In [4]:
from optbnn.gp.models.gpr import GPR
from optbnn.gp import kernels, mean_functions, priors
from optbnn.bnn.reparam_nets import GaussianMLPReparameterization
from optbnn.bnn.nets.mlp import MLP
from optbnn.bnn.likelihoods import LikGaussian
from optbnn.bnn.priors import FixedGaussianPrior, OptimGaussianPrior
from optbnn.prior_mappers.wasserstein_mapper import MapperWasserstein, WassersteinDistance
from optbnn.utils.rand_generators import MeasureSetGenerator, GridGenerator
from optbnn.utils.normalization import normalize_data
from optbnn.utils.exp_utils import get_input_range
from optbnn.metrics.sampling import compute_rhat_regression
from optbnn.metrics import uncertainty as uncertainty_metrics
from optbnn.sgmcmc_bayes_net.regression_net import RegressionNet
from optbnn.utils import util

In [ ]:
###IGNORE THIS IF YOU DON'T HAVE CUDA
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("CUDA Device Name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
print(torch.version.cuda)
print(torch.cuda.device_count())

In [ ]:
class BNN(PyroModule):
    def __init__(self, weight_prior, bias_prior, in_dim=1, out_dim=1, hid_dim=10, n_hid_layers=5):
        '''
        functional model (network architecture):
            a fully connected neural network.

        stochastic model:
            Gaussian prior on weight and bias: p(theta) ~ dist.Normal(0., weight_prior or bias_prior), where weight_prior and bias_prior are learned;
            Gaussian likelihood function: p(y | x, theta) ~ dist.Normal(functional model(x), sigma^2), where sigma ~ dist.Gamma(.5, 1).
        '''
        super().__init__()

        # make sure the dimensions are valid
        assert in_dim > 0 and out_dim > 0 and hid_dim > 0 and n_hid_layers > 0

        # activation function for the whole network, can also be ReLU or LeakyReLU
        self.activation = nn.Tanh()

        # define the layer sizes and the PyroModule layer list
        self.layer_sizes = [in_dim] + n_hid_layers * [hid_dim] + [out_dim]
        layer_list = [PyroModule[nn.Linear](self.layer_sizes[idx - 1], self.layer_sizes[idx]) for idx in
                      range(1, len(self.layer_sizes))]
        self.layers = PyroModule[torch.nn.ModuleList](layer_list)

        # set the probability distribution for each layer's weight and bias
        for layer_idx, layer in enumerate(self.layers):
            layer.weight = PyroSample(dist.Normal(0., weight_prior[layer_idx]).expand([self.layer_sizes[layer_idx + 1], self.layer_sizes[layer_idx]]).to_event(2))
            layer.bias = PyroSample(dist.Normal(0., bias_prior[layer_idx]).expand([self.layer_sizes[layer_idx + 1]]).to_event(1))

    def forward(self, x, y=None):
        # functional model(x)
        # input --> hidden
        x = self.activation(self.layers[0](x))
        # hidden --> hidden
        for layer in self.layers[1:-1]:
            x = self.activation(layer(x))
        # hidden --> output
        mu = self.layers[-1](x).squeeze()

        # sample from P(y | x, \theta)
        sigma = pyro.sample("sigma", dist.Gamma(.5, 1))
        with pyro.plate("data", x.shape[0]):
            # obs is used when quantifying and visualizing the uncertainty of predictions
            obs = pyro.sample("obs", dist.Normal(mu, sigma * sigma), obs=y)
        
        return mu

In [ ]:
def plot_predictions(preds, y):
    '''
    Function to visualize the predictions and the uncertainty of predictions.
    '''
    y_pred = preds['obs'].T.detach().numpy().mean(axis=1)
    y_std = preds['obs'].T.detach().numpy().std(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5))

    # decide the range of the y axis based on the number of the labels
    time_idx = np.array(range(len(y)))
    xlims = [time_idx.min() - 0.1, time_idx.max() + 0.1]
    # decide the range of the y axis based on the range of the labels
    ylims = [min(y.min(), y_pred.min()) - 20,
             max(y.max(), y_pred.max()) + 20]
    
    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("time", fontsize=20)
    plt.ylabel("closing price", fontsize=20)

    ax.plot(time_idx, y, 'ko', markersize=1, label="observations")
    ax.plot(time_idx, y_pred, '-', linewidth=0.5, color="#408765", label="predictive mean")
    ax.fill_between(time_idx, y_pred - 2 * y_std, y_pred + 2 * y_std, alpha=0.6, color='#86cfac', zorder=0)

    plt.legend(loc=4, fontsize=15, frameon=False)

In [ ]:
def plot_uncertainty(preds, y):
    '''
    Function to visualize only the uncertainty.
    '''
    fig, ax = plt.subplots(figsize=(10, 5))

    time_idx = np.array(range(len(y)))
    y_std = preds['obs'].T.detach().numpy().std(axis=1)

    xlims = [time_idx.min() - 0.1, time_idx.max() + 0.1]
    ylims = [y_std.min() - 0.5, y_std.max() + 0.5]

    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("time", fontsize=20)
    plt.ylabel("std of closing price", fontsize=20)

    ax.plot(time_idx, y_std, 'ko', markersize=1, label="std of predictions")
    ax.plot(time_idx, y_std, '-', linewidth=0.5, color="#408765")

    plt.legend(loc=4, fontsize=15, frameon=False)

#### BNN Prior: Dassault

In [5]:
df_pre_dassault = pd.read_csv("pregdprApril2016_Dassault.csv")
df_pre_dassault.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,target
0,255723498.0,61.03,10.0,1.0,1.163192,255723498.0,61.73,10.0,1.0,1.163192,...,61.83,10.0,1.0,1.163192,255230678.0,63.35,10.0,1.0,1.163192,61.538524
1,255723498.0,61.73,10.0,1.0,1.163192,255230678.0,62.01,10.0,1.0,1.163192,...,63.35,10.0,1.0,1.163192,255230678.0,63.41,10.0,1.0,1.163192,61.742955
2,255230678.0,62.01,10.0,1.0,1.163192,255230678.0,61.83,10.0,1.0,1.163192,...,63.41,10.0,1.0,1.163192,255230678.0,63.52,10.0,1.0,1.163192,60.854215
3,255230678.0,61.83,10.0,1.0,1.163192,255230678.0,63.35,10.0,1.0,1.163192,...,63.52,10.0,1.0,1.163192,255230678.0,63.30,10.0,1.0,1.163192,60.614367
4,255230678.0,63.35,10.0,1.0,1.163192,255230678.0,63.41,10.0,1.0,1.163192,...,63.30,10.0,1.0,1.163192,255230678.0,62.60,10.0,1.0,1.163192,60.945144


In [7]:
# Get both X and y from the DataFrame
X_pre_dassault = df_pre_dassault.iloc[:, :-1].values
y_pre_dassault = df_pre_dassault["target"].values

# Combine them into a single mask
valid_mask = ~np.isnan(X_pre_dassault).any(axis=1) & ~np.isnan(y_pre_dassault)

# Apply the same mask to both
X_pre_dassault = X_pre_dassault[valid_mask]
y_pre_dassault = y_pre_dassault[valid_mask]

In [8]:
X_pre_dassault, X_pre_dassault.shape, type(X_pre_dassault)

(array([[2.55723498e+08, 6.10300000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.16319246e+00],
        [2.55723498e+08, 6.17300000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.16319246e+00],
        [2.55230678e+08, 6.20100000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.16319246e+00],
        ...,
        [2.56725586e+08, 7.12700000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.17027004e+00],
        [2.56725586e+08, 7.13200000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.17027004e+00],
        [2.56725586e+08, 6.83100000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.17027004e+00]], shape=(269, 25)),
 (269, 25),
 numpy.ndarray)

In [9]:
y_pre_dassault, y_pre_dassault.shape, type(y_pre_dassault)

(array([61.53852442, 61.74295455, 60.8542151 , 60.61436726, 60.94514366,
        61.81897233, 64.62556665, 63.45225425, 66.8413532 , 66.9398132 ,
        68.05238175, 67.13263511, 66.51696291, 67.9197906 , 68.07606449,
        67.77384055, 67.30559209, 67.88407435, 67.37072308, 67.32655019,
        67.93183664, 67.84217944, 68.66282227, 68.34494517, 68.606806  ,
        68.87252653, 69.12276188, 69.27283807, 69.08648979, 68.7138416 ,
        69.37603734, 68.81131577, 68.87932557, 69.13480156, 69.22946182,
        69.69996498, 69.16447566, 69.02814149, 68.39400449, 67.6724961 ,
        68.30552903, 67.17130905, 67.42639018, 67.41212699, 66.77483212,
        67.16903547, 66.46530755, 66.06312233, 66.38106891, 65.9123806 ,
        65.69417283, 65.74160245, 66.44031723, 65.64432867, 65.10969189,
        64.85548837, 65.68128745, 64.79463496, 65.07602665, 65.50459542,
        64.88634277, 64.17099819, 64.46536637, 65.00492282, 64.91808325,
        64.14803501, 64.66642966, 63.92473042, 63.6

In [ ]:
noise_var = 0.1
n_units = 128
n_hidden = 1
activation_fn = "tanh"
num_iters = 300  # Number of iteterations of Wasserstein optimization
lr = 0.05        # The learning rate
n_samples = 128  # The mini-batch size
out_dir = "./exp/gdpr/optim_gaussian"

X_pre_n, y_pre_n, y_mean, y_std = normalize_data(X_pre_dassault, y_pre_dassault)
x_min, x_max = get_input_range(X_pre_n, y_pre_n)
epsilon = 1e-6
x_min = np.minimum(x_min, x_max - epsilon)
input_dim, output_dim = int(X_pre_dassault.shape[-1]), 1
    
# Initialize the measurement set generator
rand_generator = MeasureSetGenerator(X_pre_n, x_min, x_max, 0.7)

# Initialize the mean and covariance function of the target hierarchical GP prior
mean = mean_functions.Zero()
    
lengthscale = math.sqrt(2. * input_dim)
variance = 1.
kernel = kernels.RBF(input_dim=input_dim,
                     lengthscales=torch.tensor([lengthscale], dtype=torch.double),
                     variance=torch.tensor([variance], dtype=torch.double), ARD=True)

# Place hyper-priors on lengthscales and variances
kernel.lengthscales.prior = priors.LogNormal(
    torch.ones([input_dim]) * math.log(lengthscale),
    torch.ones([input_dim]) * 1.)
kernel.variance.prior = priors.LogNormal(
    torch.ones([1]) * 0.1,
    torch.ones([1]) * 1.)
        
# Initialize the GP model
gp = GPR(X=torch.from_numpy(X_pre_n), Y=torch.from_numpy(y_pre_n).reshape([-1, 1]),
             kern=kernel, mean_function=mean)
gp.likelihood.variance.set(noise_var)
    
# Initialize tunable MLP prior
hidden_dims = [n_units] * n_hidden
mlp_reparam = GaussianMLPReparameterization(input_dim, output_dim,
    hidden_dims, activation_fn, scaled_variance=True)
    
mapper = MapperWasserstein(gp, mlp_reparam, rand_generator, out_dir=out_dir,
                               output_dim=output_dim, n_data=100,
                               wasserstein_steps=(0, 300), ###more than 200
                               wasserstein_lr=0.02,
                               logger=None, wasserstein_thres=0.02,
                               n_gpu=1, gpu_gp=True) ##Change GPU if you don't have CUDA; same thing for the post training
    
w_hist = mapper.optimize(num_iters=num_iters, n_samples=n_samples,
                             lr=lr, print_every=10, save_ckpt_every=10, debug=True)

print("----" * 20)

In [ ]:
for name, param in mlp_reparam.named_parameters():
    print(f"parameter name: {name}, parameter shape: {param}")

In [ ]:
def maintain_positivity(x):
    '''
    maintain the positivity of weight and bias standard derivations
    '''
    return np.log(1 + np.exp(x))

pre_weight_prior = [maintain_positivity(4.3945), maintain_positivity(1.2164)]
pre_bias_prior = [maintain_positivity(3.0868), maintain_positivity(-2.0868)]

In [ ]:
# clear parameters to ensure every training start from scratch
pyro.clear_param_store()

# set up BNN
model_VI = BNN(pre_weight_prior, pre_bias_prior, in_dim=25, out_dim=1, hid_dim=128, n_hid_layers=1)

#mean_field_guide = AutoDiagonalNormal(model_VI) # mean field variational inference
guide = AutoMultivariateNormal(model_VI) # use multivariate normal with full covariance to approxiamte posterior

# apply SGD to maximizing the ELBO
optimizer = pyro.optim.Adam({"lr": 0.001})
svi = SVI(model_VI, guide, optimizer, loss=Trace_ELBO())

# # clear parameters to avoid influencing others
pyro.clear_param_store()

In [ ]:
%time
num_epochs = 10000 # number of training epoches, 10000 now for quick test: 25000
progress_bar = trange(num_epochs) # show progress bar (only for visualization purpose)

X_pre_n_tensor = torch.tensor(X_pre_n, dtype=torch.float)
y_pre_n_tensor = torch.tensor(y_pre_n, dtype=torch.float)

for epoch in progress_bar:
    loss = svi.step(X_pre_n_tensor, y_pre_n_tensor)
    progress_bar.set_postfix(loss=f"{loss / X_pre_n.shape[0]:.3f}")
    if epoch % 1000 == 0: ##change 1000
        print("[iteration %04d] loss: %.3f" % (epoch + 1, loss / X_pre_n.shape[0])) ##36.699

In [ ]:
predictive = Predictive(model_VI, guide=guide, num_samples=1000)
preds = predictive(X_pre_n_tensor)
plot_predictions(preds, y_pre_n_tensor)

In [ ]:
##RMSE
pred_samples = preds["obs"]
pred_mean = pred_samples.mean(dim=0)
# Calculate RMSE
y_true = y_pre_n_tensor
rmse = torch.sqrt(torch.mean((pred_mean - y_true) ** 2))
print(rmse)

In [ ]:
plot_uncertainty(preds, y_pre_n_tensor)

### BNN Posterior: Dassault

In [10]:
df_post_dassault = pd.read_csv("postgdprMay2018_Dassault.csv")
df_post_dassault.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,target
0,260506586.0,120.65,10.0,1.0,1.191664,260506586.0,121.45,10.0,1.0,1.191664,...,120.30,10.0,1.0,1.191664,260506586.0,118.75,10.0,1.0,1.191664,72.463260
1,260506586.0,121.45,10.0,1.0,1.191664,260506586.0,119.85,10.0,1.0,1.191664,...,118.75,10.0,1.0,1.191664,260506586.0,118.95,10.0,1.0,1.191664,72.526236
2,260506586.0,119.85,10.0,1.0,1.191664,260506586.0,120.30,10.0,1.0,1.191664,...,118.95,10.0,1.0,1.191664,260506586.0,118.55,10.0,1.0,1.191664,74.508429
3,260506586.0,120.30,10.0,1.0,1.191664,260506586.0,118.75,10.0,1.0,1.191664,...,118.55,10.0,1.0,1.191664,260506586.0,119.35,10.0,1.0,1.191664,74.502323
4,260506586.0,118.75,10.0,1.0,1.191664,260506586.0,118.95,10.0,1.0,1.191664,...,119.35,10.0,1.0,1.191664,260506586.0,121.85,10.0,1.0,1.191664,75.535832


In [12]:
##Extract Values
X_post_dassault = df_post_dassault.iloc[:, :-1].values
y_post_dassault = df_post_dassault["target"].values

# Create a mask for rows without NaNs in either X or y
valid_mask = ~np.isnan(X_post_dassault).any(axis=1) & ~np.isnan(y_post_dassault)

# Apply mask to both X and y
X_post_dassault = X_post_dassault[valid_mask]
y_post_dassault = y_post_dassault[valid_mask]

In [13]:
X_post_dassault, X_post_dassault.shape, type(X_post_dassault)

(array([[2.60506586e+08, 1.20650000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.19166443e+00],
        [2.60506586e+08, 1.21450000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.19166443e+00],
        [2.60506586e+08, 1.19850000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.19166443e+00],
        ...,
        [2.62927968e+08, 1.43150000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.19752138e+00],
        [2.62927968e+08, 1.43250000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.19752138e+00],
        [2.62927968e+08, 1.43850000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.19752138e+00]], shape=(398, 25)),
 (398, 25),
 numpy.ndarray)

In [14]:
y_post_dassault, y_post_dassault.shape, type(y_post_dassault)

(array([72.46326003, 72.5262357 , 74.50842943, 74.50232256, 75.53583187,
        76.57507706, 76.39268269, 76.37525899, 75.25164013, 73.45801233,
        73.73675369, 73.97438282, 74.59845874, 73.82270145, 73.0707952 ,
        72.70803648, 72.66400094, 71.98984022, 72.62764333, 72.74684127,
        72.83976136, 74.35947857, 75.28881848, 76.52861812, 76.49438918,
        77.15176437, 77.89188218, 77.7023347 , 77.50102448, 76.5093257 ,
        75.52173412, 75.24501902, 73.48477162, 75.39237725, 75.40275557,
        74.40197061, 74.17470031, 74.25358222, 75.90373332, 75.019253  ,
        75.0586543 , 73.46307012, 74.46146683, 75.18833002, 75.90962332,
        77.0081488 , 78.44104714, 78.24649973, 78.34005823, 78.1356668 ,
        79.35384873, 78.55931344, 79.02810668, 80.10896386, 80.29243862,
        81.58037554, 80.90179801, 80.80184494, 81.73978026, 82.35795575,
        82.33427182, 80.42635533, 77.66465211, 77.14932824, 77.00010139,
        77.06842568, 76.82347083, 77.1600428 , 76.5

In [ ]:
noise_var = 0.1
n_units = 128
n_hidden = 1
activation_fn = "tanh"
num_iters = 300  # Number of iteterations of Wasserstein optimization
lr = 0.05        # The learning rate
n_samples = 128  # The mini-batch size
out_dir = "./exp/gdpr/optim_gaussian"

X_post_n, y_post_n, y_mean, y_std = normalize_data(X_post_dassault, y_post_dassault)
x_min, x_max = get_input_range(X_post_n, X_post_n)
epsilon = 1e-6
x_min = np.minimum(x_min, x_max - epsilon)
input_dim, output_dim = int(X_post_dassault.shape[-1]), 1
    
# Initialize the measurement set generator
rand_generator = MeasureSetGenerator(X_post_n, x_min, x_max, 0.7)
    
# Initialize the mean and covariance function of the target hierarchical GP prior
mean = mean_functions.Zero()
    
lengthscale = math.sqrt(2. * input_dim)
variance = 1.
kernel = kernels.RBF(input_dim=input_dim,
                     lengthscales=torch.tensor([lengthscale], dtype=torch.double),
                     variance=torch.tensor([variance], dtype=torch.double), ARD=True)

# Place hyper-priors on lengthscales and variances
kernel.lengthscales.prior = priors.LogNormal(
    torch.ones([input_dim]) * math.log(lengthscale),
    torch.ones([input_dim]) * 1.)
kernel.variance.prior = priors.LogNormal(
    torch.ones([1]) * 0.1,
    torch.ones([1]) * 1.)
        
# Initialize the GP model
gp = GPR(X=torch.from_numpy(X_post_n), Y=torch.from_numpy(y_post_n).reshape([-1, 1]),
             kern=kernel, mean_function=mean)
gp.likelihood.variance.set(noise_var)
    
# Initialize tunable MLP prior
hidden_dims = [n_units] * n_hidden
mlp_reparam = GaussianMLPReparameterization(input_dim, output_dim,
    hidden_dims, activation_fn, scaled_variance=True)
    
mapper = MapperWasserstein(gp, mlp_reparam, rand_generator, out_dir=out_dir,
                               output_dim=output_dim, n_data=100,
                               wasserstein_steps=(0, 300), ##Should be more than 200
                               wasserstein_lr=0.02,
                               logger=None, wasserstein_thres=0.1,
                               n_gpu=1, gpu_gp=True) ##Change GPU if you don't have CUDA; same thing for the PRE training
    
w_hist = mapper.optimize(num_iters=num_iters, n_samples=n_samples,
                             lr=lr, print_every=10, save_ckpt_every=10, debug=True)

print("----" * 20)

In [ ]:
for name, param in mlp_reparam.named_parameters():
    print(f"parameter name: {name}, parameter shape: {param}")

In [ ]:
def maintain_positivity(x):
    '''
    maintain the positivity of weight and bias standard derivations
    '''
    return np.log(1 + np.exp(x))

post_weight_prior = [maintain_positivity(3.3871), maintain_positivity(1.1624)]
post_bias_prior = [maintain_positivity(2.3243), maintain_positivity(-0.4564)]

In [ ]:
# clear parameters to ensure every training start from scratch
pyro.clear_param_store()

# set up BNN
model_VI = BNN(post_weight_prior, post_bias_prior, in_dim=25, out_dim=1, hid_dim=128, n_hid_layers=1)

#mean_field_guide = AutoDiagonalNormal(model_VI) # mean field variational inference
guide = AutoMultivariateNormal(model_VI) # use multivariate normal with full covariance to approxiamte posterior

# apply SGD to maximizing the ELBO
optimizer = pyro.optim.Adam({"lr": 0.001})
svi = SVI(model_VI, guide, optimizer, loss=Trace_ELBO())

# # clear parameters to avoid influencing others
pyro.clear_param_store()

In [ ]:
num_epochs = 20001 # number of training epoches, 10000 now for quick test
progress_bar = trange(num_epochs) # show progress bar (only for visualization purpose)

X_post_n_tensor = torch.tensor(X_post_n, dtype=torch.float)
y_post_n_tensor = torch.tensor(y_post_n, dtype=torch.float)

for epoch in progress_bar:
    loss = svi.step(X_post_n_tensor, y_post_n_tensor)
    progress_bar.set_postfix(loss=f"{loss / X_post_n.shape[0]:.3f}")
    if epoch % 1000 == 0:
        print("[iteration %04d] loss: %.3f" % (epoch + 1, loss / X_post_n.shape[0])) #11.525

In [ ]:
predictive = Predictive(model_VI, guide=guide, num_samples=1000)
preds = predictive(X_post_n_tensor)
plot_predictions(preds, y_post_n_tensor)

In [ ]:
##RMSE
pred_samples = preds["obs"]
pred_mean = pred_samples.mean(dim=0)
# Calculate RMSE
y_true = y_post_n_tensor
rmse = torch.sqrt(torch.mean((pred_mean - y_true) ** 2))
print(rmse)

In [ ]:
plot_uncertainty(preds, y_post_n_tensor)